# Google Search Questions Explorer

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook maps the public question landscape around any topic by collecting what people actually search for on Google. Enter a topic, and the notebook generates hundreds of real search suggestions organized by question type — what, why, how, who, comparisons, and more — exported as structured data for discourse analysis.

The notebook uses Google's public Autocomplete API to retrieve search suggestions. This is the same data that appears when you start typing in the Google search bar. No API key is required, and the data reflects real search behavior.

## Key Features

1. **No API Key Required**: Uses Google's public Autocomplete API directly
2. **Question Mapping**: Systematically explores what, why, how, who, where, and when questions about your topic
3. **Alphabet Expansion**: Discovers additional queries by appending each letter of the alphabet to your topic
4. **Research-Oriented Prefixes**: Includes academic prefixes like "methods for," "examples of," "history of"
5. **Country & Language Selection**: Target suggestions from specific regions
6. **Structured Export**: Download deduplicated results as CSV or Excel

## Workflow

1. **Configure**: Enter your research topic and select country/language
2. **Fetch**: Retrieve search suggestions across question types and alphabet expansions
3. **Review**: Preview the question landscape in a table
4. **Export**: Download structured data for qualitative analysis

## How It Works

The notebook queries Google Autocomplete with your topic combined with different prefixes:

- **Question words**: "what is [topic]", "why do [topic]", "how to [topic]"
- **Research prefixes**: "methods for [topic]", "examples of [topic]", "history of [topic]"
- **Comparisons**: "[topic] vs", "difference between [topic]"
- **Alphabet expansion**: "[topic] a", "[topic] b", ... "[topic] z"

Each query returns up to 10 suggestions, and results are deduplicated. A typical search produces 200–400 unique suggestions.

## Applications

This notebook supports research that involves understanding public knowledge, discourse, and curiosity about a topic — mapping what questions people ask, identifying gaps in public understanding, tracking how topics are framed in search behavior, and generating interview questions grounded in public discourse. Useful for discourse analysis, media studies, public anthropology, and research design.

## Methodological Positioning

This notebook represents a **computational anthropology** approach — using search behavior data as a window into public discourse and knowledge. Google Autocomplete reflects aggregated search patterns, not individual searches, and is influenced by personalization, location, and trending topics. Interpret as indicative of search behavior, not representative of public opinion.

**Important**: This notebook retrieves aggregated search suggestion data. It does not track individuals or access private information.

## Target Audience

Designed for anthropologists and qualitative researchers exploring public discourse — from graduate students scoping research topics to applied researchers mapping public knowledge landscapes.

## Technical Approach

The notebook queries Google's Autocomplete API (`suggestqueries.google.com/complete/search`), which is the same endpoint that powers search bar suggestions. It supports language and country parameters. All processing runs locally in the notebook.

## License & Attribution

This notebook is released under the **CC BY-NC 4.0** license. You may adapt and share it for non-commercial purposes with proper attribution.



## Citation

If you use this toolkit in your academic research, please cite:

> Artz, Matt. 2025. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Human Organization. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

## Setup and Installation

Install required Python packages and import libraries for Google Autocomplete retrieval, data processing, and interactive configuration. Run this cell first to ensure all dependencies are available.

In [ ]:
# Install required packages
!pip install pandas openpyxl ipywidgets requests -q

import re
import unicodedata
import requests
import pandas as pd
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore')

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

AUTOCOMPLETE_URL = 'https://suggestqueries.google.com/complete/search'


def get_suggestions(query, lang='en', country='us'):
    """Fetch autocomplete suggestions from Google."""
    params = {'client': 'firefox', 'q': query, 'hl': lang, 'gl': country}
    r = requests.get(AUTOCOMPLETE_URL, params=params, timeout=10)
    return r.json()[1]


def normalize_text(text):
    """Normalize unicode characters."""
    if not isinstance(text, str):
        return text
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\u2011', '-').replace('\u2013', '-').replace('\u2014', '-')
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = text.replace('\u2026', '...')
    return text


def make_slug(query):
    """Create a clean filename slug."""
    slug = re.sub(r'[^a-zA-Z0-9]+', '_', query).strip('_')[:30]
    return slug if slug else 'search'


def style_excel(filepath):
    """Apply branded styling to an Excel file."""
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    wb = load_workbook(filepath)
    header_fill = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF')
    thin_border = Border(
        left=Side(style='thin', color='E7ECEF'), right=Side(style='thin', color='E7ECEF'),
        top=Side(style='thin', color='E7ECEF'), bottom=Side(style='thin', color='E7ECEF'),
    )
    for ws in wb.worksheets:
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = thin_border
        for col in ws.columns:
            max_len = max(len(str(c.value or '')) for c in col)
            ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 10), 60)
        ws.freeze_panes = 'A2'
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(vertical='top', wrap_text=True)
                cell.border = thin_border
    wb.save(filepath)


print("\u2713 All packages installed and libraries loaded successfully")
print(f"\u2713 Environment: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print("\U0001f50d Ready to configure your search question exploration!")

## Search Configuration

Configure your topic and preferences using the interactive controls below. The notebook will systematically explore what people search for about your topic.

In [ ]:
# Interactive Search Interface

style = {'description_width': '120px'}
layout = widgets.Layout(width='500px')

instructions_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
instructions_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f50d Google Search Questions Explorer</h2>'
instructions_html += '<p><strong>Welcome!</strong> This notebook maps the public question landscape around any topic using Google Autocomplete data.</p>'
instructions_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
instructions_html += '<ol>'
instructions_html += '<li><strong>Topic:</strong> Enter your research topic below</li>'
instructions_html += '<li><strong>Options:</strong> Select which question types to explore</li>'
instructions_html += '<li><strong>Fetch:</strong> Click to collect search suggestions (typically 200\u2013400 results)</li>'
instructions_html += '<li><strong>Export:</strong> Download as CSV or Excel</li>'
instructions_html += '</ol>'
instructions_html += '</div>'

instructions = widgets.HTML(value=instructions_html)

topic_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4ac Topic</h3>')

topic_input = widgets.Text(
    value='',
    placeholder='Enter your research topic (e.g., "ethnography", "climate migration")',
    description='Topic:',
    layout=layout,
    style=style
)

options_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\u2699\ufe0f Options</h3>')

question_chk = widgets.Checkbox(value=True, description='Question words (what, why, how, who, where, when)', indent=False, style={'description_width': 'auto'})
research_chk = widgets.Checkbox(value=True, description='Research prefixes (methods, examples, history, types, importance)', indent=False, style={'description_width': 'auto'})
comparison_chk = widgets.Checkbox(value=True, description='Comparisons (vs, difference between)', indent=False, style={'description_width': 'auto'})
alphabet_chk = widgets.Checkbox(value=True, description='Alphabet expansion (topic + a, b, c, ... z)', indent=False, style={'description_width': 'auto'})

country_input = widgets.Dropdown(
    options=[
        ('United States', 'us'), ('United Kingdom', 'gb'), ('Canada', 'ca'),
        ('Australia', 'au'), ('India', 'in'), ('Germany', 'de'),
        ('France', 'fr'), ('Brazil', 'br'), ('Mexico', 'mx'), ('Japan', 'jp'),
    ],
    value='us',
    description='Country:',
    layout=layout,
    style=style
)

lang_input = widgets.Dropdown(
    options=[
        ('English', 'en'), ('Spanish', 'es'), ('French', 'fr'),
        ('German', 'de'), ('Portuguese', 'pt'), ('Japanese', 'ja'),
    ],
    value='en',
    description='Language:',
    layout=layout,
    style=style
)

export_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c1 Export</h3>')

format_input = widgets.Dropdown(
    options=[('CSV (.csv)', 'csv'), ('Excel (.xlsx)', 'xlsx')],
    value='csv',
    description='Format:',
    layout=layout,
    style=style
)

fetch_btn = widgets.Button(
    description='Explore Questions',
    button_style='',
    layout=widgets.Layout(width='200px', margin='20px 0'),
    style={'button_color': '#6096BA'}
)

progress_label = widgets.HTML(value='')
out = widgets.Output()


QUESTION_PREFIXES = [
    'what is', 'what are', 'what does',
    'why is', 'why do', 'why does',
    'how to', 'how does', 'how do',
    'who studies', 'who uses',
    'where is', 'where do',
    'when did', 'when was',
]

RESEARCH_PREFIXES = [
    'methods for', 'examples of', 'types of',
    'history of', 'importance of', 'definition of',
    'theory of', 'criticism of', 'future of',
]

COMPARISON_PREFIXES = [
    'vs', 'difference between',
]


def on_fetch_clicked(_):
    out.clear_output()
    progress_label.value = ''

    topic = topic_input.value.strip()
    if not topic:
        with out:
            print("\u26a0\ufe0f Please enter a research topic.")
        return

    lang = lang_input.value
    country = country_input.value

    # Build list of queries
    queries = []

    if question_chk.value:
        for prefix in QUESTION_PREFIXES:
            queries.append(('Question', prefix, f'{prefix} {topic}'))

    if research_chk.value:
        for prefix in RESEARCH_PREFIXES:
            queries.append(('Research', prefix, f'{prefix} {topic}'))

    if comparison_chk.value:
        for prefix in COMPARISON_PREFIXES:
            if prefix == 'vs':
                queries.append(('Comparison', prefix, f'{topic} vs'))
            else:
                queries.append(('Comparison', prefix, f'{prefix} {topic}'))

    if alphabet_chk.value:
        for letter in 'abcdefghijklmnopqrstuvwxyz':
            queries.append(('Alphabet', f'{topic} {letter}...', f'{topic} {letter}'))

    # Also get plain topic suggestions
    queries.insert(0, ('Direct', topic, topic))

    with out:
        print(f"\U0001f50d Exploring search questions for: {topic}")
        print(f"   Country: {country}, Language: {lang}")
        print(f"   Queries to run: {len(queries)}")
        print()

    all_rows = []
    total = len(queries)

    for i, (category, prefix, query) in enumerate(queries):
        progress_label.value = f'<span style="color: #274C77;">\u2713 {i+1}/{total} queries...</span>'
        try:
            suggestions = get_suggestions(query, lang=lang, country=country)
            for s in suggestions:
                all_rows.append({
                    'category': category,
                    'prefix': prefix,
                    'suggestion': normalize_text(s),
                })
        except Exception:
            pass
        # Small delay to be respectful
        if i % 10 == 9:
            time.sleep(0.2)

    progress_label.value = ''

    if not all_rows:
        with out:
            print("\u26a0\ufe0f No suggestions found. Try a different topic.")
        return

    df = pd.DataFrame(all_rows).drop_duplicates(subset=['suggestion'])

    with out:
        print(f"\u2713 Collected {len(df)} unique search suggestions")
        print()

        # Summary by category
        print("\U0001f4ca Summary by category:")
        for cat, count in df['category'].value_counts().items():
            print(f"   {cat}: {count} suggestions")
        print()

        print("\U0001f4cb Preview:")
        display(df.head(20))

    # Export
    try:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        slug = make_slug(topic)
        base = f'search_questions_{slug}_{timestamp}'
        fmt = format_input.value

        if fmt == 'xlsx':
            filepath = f'{base}.xlsx'
            df.to_excel(filepath, sheet_name='Search Questions', index=False, engine='openpyxl')
            style_excel(filepath)
        else:
            filepath = f'{base}.csv'
            df.to_csv(filepath, index=False)

        with out:
            print()
            print(f"\u2713 Saved: {filepath} ({len(df)} suggestions)")
            if IN_COLAB:
                colab_files.download(filepath)
    except Exception as e:
        with out:
            print(f"\u274c Export error: {type(e).__name__}: {e}")



fetch_btn.on_click(on_fetch_clicked)

display(widgets.VBox([
    instructions,
    topic_header,
    topic_input,
    options_header,
    question_chk,
    research_chk,
    comparison_chk,
    alphabet_chk,
    country_input,
    lang_input,
    export_header,
    format_input,
    fetch_btn,
    progress_label,
    out,
]))